# Phase 2 — LLC Calibration (interactive)

Run **after** Phase 1 training is complete.

**Kaggle:** Add the Phase 1 notebook's output version as a dataset input
(notebook settings → Add data → your notebook → version N).
Then run cells one by one (Shift+Enter) — **do not** Save and Run All,
because you need to inspect the chain trace plot before filling in the save cell.

**Takes ≈ 20 min on T4.** No GPU strictly required but speeds up the SGLD chains.

## Section 0 — Setup

In [ ]:
import os, sys, shutil, subprocess, glob

PLATFORM = "kaggle"   # "kaggle" or "colab"
REPO_URL  = "https://github.com/makataomu/slt-diplomka"

if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
    REPO_DIR    = "/content/slt"
    PERSIST_DIR = "/content/drive/MyDrive/slt_persist"
    for d in ["results/checkpoints", "results/metrics", "results/figures"]:
        os.makedirs(f"{PERSIST_DIR}/{d}", exist_ok=True)
else:
    REPO_DIR    = "/kaggle/working/slt"
    PERSIST_DIR = None

if os.path.exists(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("Pulled latest from GitHub")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Cloned from GitHub")

if PLATFORM == "colab":
    lnk = f"{REPO_DIR}/results"
    if os.path.islink(lnk): os.unlink(lnk)
    elif os.path.isdir(lnk): shutil.rmtree(lnk)
    os.symlink(f"{PERSIST_DIR}/results", lnk)
    print(f"results/ -> {PERSIST_DIR}/results")
else:
    for sub in ["results/checkpoints", "results/metrics", "results/figures"]:
        os.makedirs(f"{REPO_DIR}/{sub}", exist_ok=True)
    # Handles nested notebook output paths like
    # /kaggle/input/notebooks/{user}/{name}/slt/results/
    for prev in glob.glob("/kaggle/input/**/slt/results", recursive=True):
        print(f"Restoring from {prev} ...")
        for sub in ["checkpoints", "metrics"]:
            src, dst = f"{prev}/{sub}", f"{REPO_DIR}/results/{sub}"
            if os.path.exists(src):
                for item in os.listdir(src):
                    s, d = f"{src}/{item}", f"{dst}/{item}"
                    if not os.path.exists(d):
                        (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
        print("  Done.")

# Restore calibration yaml if present in any dataset input
calib_local = f"{REPO_DIR}/configs/llc_calibration.yaml"
if PLATFORM == "colab":
    src = f"{PERSIST_DIR}/llc_calibration.yaml"
    if os.path.exists(src): shutil.copy(src, calib_local); print("Restored calib yaml")
else:
    hits = glob.glob("/kaggle/input/**/llc_calibration.yaml", recursive=True)
    if hits: shutil.copy(hits[0], calib_local); print(f"Restored calib yaml from {hits[0]}")

os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
print(f"\nReady. Platform={PLATFORM} | cwd={os.getcwd()}")

# Verify calibration checkpoints exist (use latest)
from pathlib import Path
ckpt_dir = Path("results/checkpoints/ratio_0.50/seed_0")
ckpts = sorted(ckpt_dir.glob("epoch_*.pt"), key=lambda p: int(p.stem.split('_')[1]))
if ckpts:
    print(f"Calibration checkpoint: ✓ found {ckpts[-1].name} ({len(ckpts)} total)")
else:
    print("✗ MISSING — add Phase 1 output as dataset input")


## Section 1 — Install dependencies

In [ ]:
%pip install -q transformer_lens devinterp zarr==3.1.6

import importlib.metadata, torch
print(f"devinterp {importlib.metadata.version('devinterp')}  "
      f"| torch {torch.__version__}  "
      f"| device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


## Section 2 — Run calibration

Uses the final checkpoint of `ratio=0.50 seed=0`.
Runs 8 SGLD chains × 500 draws — prints per-chain stats when done.

In [ ]:
# Calibrate on the FINAL checkpoint (burnin=500 is the new default in the script)
!python src/llc_estimation.py --ratio 0.50 --seed 0 --calibrate


## Section 3 — Inspect chain traces

**Good calibration:** all chains fluctuate in a stable band — no divergence, no flatline.
- Chains diverge upward → reduce `epsilon` (e.g. `1e-5`)
- Chains flatline / don't mix → increase `epsilon` or `num_draws`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

traces = np.load("results/metrics/calibration_traces.npy")
print(f"Traces shape: {traces.shape}  (chains × draws)")
for i, c in enumerate(traces):
    print(f"  Chain {i}: mean={c.mean():.4f}  std={c.std():.4f}  "
          f"min={c.min():.4f}  max={c.max():.4f}")

fig, ax = plt.subplots(figsize=(11, 4))
for i, c in enumerate(traces):
    ax.plot(c, lw=0.8, alpha=0.75, label=f"Chain {i}")
ax.set(xlabel="Draw", ylabel="Loss (SGLD)",
       title="Calibration chains — should mix in a stable band")
ax.legend(fontsize=8, ncol=4)
plt.tight_layout(); plt.show()


## Section 3.5 — Stability check at an early checkpoint

Run the same calibration on epoch ~1000 (mid-training, during the LLC dip).
**Why:** the SGLD hyperparams were found at the final checkpoint where the loss
landscape is flat and wide. At epoch 1000 the landscape is different (possibly
sharper). If chains diverge here with the same epsilon, the LLC values during
the early-epoch dip in Phase 3 may be calibration artifacts.

**Good:** chains mix in a stable band at both early and final checkpoints → same
epsilon works throughout; the Phase 3 sweep is trustworthy.
**Bad (chains blow up at epoch 1000):** reduce epsilon by 10x and note in calibration_notes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- run calibration on epoch ~1000 ---
!python src/llc_estimation.py --ratio 0.50 --seed 0 --calibrate --checkpoint_epoch 1000

# --- load and plot both traces side by side ---
from pathlib import Path

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

for ax, label in zip(axes, ["early (~epoch 1000)", "final (~epoch 10000)"]):
    # find the matching file
    candidates = sorted(Path("results/metrics").glob("calibration_traces_epoch*.npy"))
    if label.startswith("early"):
        # pick the smallest epoch file
        f = min(candidates, key=lambda p: int(p.stem.replace("calibration_traces_epoch", "")))
    else:
        f = max(candidates, key=lambda p: int(p.stem.replace("calibration_traces_epoch", "")))
    traces = np.load(f)
    for i, chain in enumerate(traces):
        ax.plot(chain, lw=0.8, alpha=0.75, label=f"Chain {i}")
    ax.set(xlabel="Draw", ylabel="Loss (SGLD)",
           title=f"Chains — {label}\n{f.name}")
    ax.legend(fontsize=7, ncol=4)

plt.tight_layout()
plt.savefig("results/figures/fig_calibration_traces.pdf", bbox_inches="tight")
plt.show()
print("Saved results/figures/fig_calibration_traces.pdf")


## Section 4 — Save calibrated hyperparams

**Edit the values below** based on what you saw in the trace plot, then run the cell.
Start from `nbeta=46.2` (= `default_nbeta(256)`) and `epsilon=1e-4`.

In [ ]:
import yaml
from pathlib import Path
from datetime import date

# ── EDIT THESE after inspecting traces ───────────────────────────────────────
CALIBRATED = dict(
    calibrated            = True,
    epsilon               = 1e-4,
    nbeta                 = 46.2,   # default_nbeta(256) = 256/log(256)
    gamma                 = 10.0,
    num_chains            = 8,
    num_draws             = 500,
    num_burnin_steps      = 500,    # increased from 100 — chain 7 needed longer burn-in
    calibration_checkpoint= str(sorted(Path("results/checkpoints/ratio_0.50/seed_0").glob("epoch_*.pt"), key=lambda p: int(p.stem.split("_")[1]))[-1]),
    calibration_date      = str(date.today()),
    calibration_notes     = "",  # e.g. "chains stable at both epoch 1000 and 10000"
)
# ─────────────────────────────────────────────────────────────────────────────

yaml_str = yaml.dump(CALIBRATED, default_flow_style=False)
Path("configs/llc_calibration.yaml").write_text(yaml_str)

# Back up so Phase 3 can restore it
if PLATFORM == "colab":
    import shutil
    shutil.copy("configs/llc_calibration.yaml",
                f"{PERSIST_DIR}/llc_calibration.yaml")
    print(f"Backed up to Drive")
else:  # kaggle
    import shutil
    shutil.copy("configs/llc_calibration.yaml",
                "/kaggle/working/llc_calibration.yaml")
    print("Saved to /kaggle/working/llc_calibration.yaml (appears in session output)")

print("\n" + yaml_str)
